# Pünktlichkeit am Frankfurter Hauptbahnhof

Analyse der Verspätungen der Deutschen Bahn am Frankfurt (Main) Hbf in der Woche vom 6. bis 12. Oktober 2025.

**Fragen:**
1. Wie viele Züge sind pünktlich?
2. Welche Zuggruppe ist am unpünktlichsten?
3. Zu welcher Uhrzeit gibt es die meisten Verspätungen?

**Daten:** piebro/deutsche-bahn-data (HuggingFace), Oktober 2025, Lizenz CC BY 4.0

**Technologies:** Python, DuckDB (SQL)

In [1]:
import duckdb
print("Hallo, DuckDb funktioniert gut.")

Hallo, DuckDb funktioniert gut.


## 1. Daten laden und erster Überblick

Die Datei enthält alle Halte an ca. 100 großen Bahnhöfen im Oktober 2025. Jede Zeile ist ein Halt eines Zuges an einem Bahnhof.


In [2]:
duckdb.sql("SELECT COUNT(*) FROM '../data/data-2025-10.parquet'")


┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      1989180 │
└──────────────┘

In [3]:
duckdb.sql("select * from '../data/data-2025-10.parquet' limit 5")


┌──────────────────────┬──────────────────────┬──────────┬──────────────┬─────────────┬───────────────────────────┬──────────────┬─────────────────────┬─────────────────────┬───────────────────────┬────────────┬──────────────────────┬────────────────────────┬──────────────────────┬─────────────────────┬────────────────────────┬───────────────────────┬────────────────────────────────────┐
│     station_name     │   xml_station_name   │   eva    │ train_number │ line_number │ final_destination_station │ delay_in_min │        time         │ arrival_is_canceled │ departure_is_canceled │ train_type │  train_line_ride_id  │ train_line_station_num │ arrival_planned_time │ arrival_change_time │ departure_planned_time │ departure_change_time │                 id                 │
│       varchar        │       varchar        │ varchar  │   varchar    │   varchar   │          varchar          │    int32     │    timestamp_ns     │       boolean       │        boolean        │  varchar   │       

In [4]:
duckdb.sql("""
    select station_name, count(*) as anzahl
    from '../data/data-2025-10.parquet'
    group by station_name
    order by  anzahl DESC
    limit 20

""")

┌────────────────────────────┬────────┐
│        station_name        │ anzahl │
│          varchar           │ int64  │
├────────────────────────────┼────────┤
│ Berlin Ostkreuz            │  64476 │
│ München Hbf                │  64355 │
│ Frankfurt (Main) Hbf       │  60842 │
│ Hamburg Hbf                │  59866 │
│ Berlin Friedrichstraße     │  49005 │
│ Stuttgart Hbf              │  48001 │
│ Düsseldorf Hbf             │  45821 │
│ Berlin Hauptbahnhof        │  44992 │
│ Berlin Ostbahnhof          │  40509 │
│ Köln Hbf                   │  38906 │
│ Berlin Gesundbrunnen       │  37814 │
│ Hamburg-Altona             │  37613 │
│ Berlin Südkreuz            │  36477 │
│ München Ost                │  34648 │
│ Dortmund Hbf               │  33513 │
│ Köln Messe/Deutz           │  32614 │
│ Nürnberg Hbf               │  32068 │
│ München-Pasing             │  31498 │
│ Berlin Zoologischer Garten │  30544 │
│ Hannover Hbf               │  30531 │
└────────────────────────────┴────────┘


## 2. Auswahl der Daten

Für die Analyse wird eine View `frankfurt` erstellt:
- nur **Frankfurt (Main) Hbf**
- nur die Woche **Montag, 6. bis Sonntag, 12. Oktober 2025**

Zusätzliche Spalten:
- `datum` und `wochentag`: um die Ergebnisse Tag für Tag zu vergleichen
- `zuggruppe`: fasst die Zugtypen zu Fernverkehr, Regionalverkehr und S-Bahn zusammen


In [5]:
duckdb.sql("""
      CREATE OR REPLACE VIEW frankfurt AS
      SELECT
           *,
           CAST(time AS DATE) AS datum,
           dayname(time) AS wochentag,
           CASE
               WHEN train_type IN ('ICE', 'IC', 'EC', 'ECE', 'TGV','NJ' ) THEN 'Fernverkehr'
               WHEN train_type IN ('RE', 'RB', 'HLB', 'VIA', 'STN' ) THEN 'Regionalverkehr'
               WHEN train_type = 'S' THEN 'S-Bahn'
               ELSE 'Sonstige'
           END AS zuggruppe           
      FROM '../data/data-2025-10.parquet'
      WHERE station_name = 'Frankfurt (Main) Hbf'
            and time >= '2025-10-06'  
            and time <  '2025-10-13'  


""")

In [6]:
duckdb.sql("""
      SELECT zuggruppe, COUNT(*) AS anzahl
      FROM frankfurt
      GROUP BY zuggruppe
      ORDER BY anzahl DESC

""")

┌─────────────────┬────────┐
│    zuggruppe    │ anzahl │
│     varchar     │ int64  │
├─────────────────┼────────┤
│ S-Bahn          │   5979 │
│ Regionalverkehr │   5213 │
│ Fernverkehr     │   1863 │
└─────────────────┴────────┘

## 3. Datenqualität und Überblick

Bevor die Fragen beantwortet werden, wird geprüft, ob es Daten gibt, die nicht mitgezählt werden dürfen: ausgefallene Halte, Busse und unrealistische Werte.


In [7]:
duckdb.sql("""
     select
       count(*) as gesamt,
       sum(case when arrival_is_canceled or departure_is_canceled then 1 else 0 end) as ausgefallen,
       sum(case when train_type = 'Bus' then 1 else 0 end) as busse,
       sum(case when delay_in_min < 0 then 1 else 0 end) as zu_frueh
    from frankfurt

""")

┌────────┬─────────────┬────────┬──────────┐
│ gesamt │ ausgefallen │ busse  │ zu_frueh │
│ int64  │   int128    │ int128 │  int128  │
├────────┼─────────────┼────────┼──────────┤
│  13055 │         552 │      0 │      380 │
└────────┴─────────────┴────────┴──────────┘

In [8]:
duckdb.sql("""
     SELECT train_type, COUNT(*) AS anzahl
     FROM frankfurt
     GROUP BY train_type
     ORDER BY anzahl DESC 
""")

┌────────────┬────────┐
│ train_type │ anzahl │
│  varchar   │ int64  │
├────────────┼────────┤
│ S          │   5979 │
│ RE         │   1780 │
│ ICE        │   1636 │
│ RB         │   1582 │
│ HLB        │    933 │
│ VIA        │    878 │
│ IC         │    144 │
│ TGV        │     42 │
│ STN        │     40 │
│ EC         │     14 │
│ NJ         │     14 │
│ ECE        │     13 │
└────────────┴────────┘
  12 rows   2 columns

In [9]:
duckdb.sql("DESCRIBE frankfurt")

┌───────────────────────────┬──────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name        │ column_type  │  null   │   key   │ default │  extra  │
│          varchar          │   varchar    │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────┼──────────────┼─────────┼─────────┼─────────┼─────────┤
│ station_name              │ VARCHAR      │ YES     │ NULL    │ NULL    │ NULL    │
│ xml_station_name          │ VARCHAR      │ YES     │ NULL    │ NULL    │ NULL    │
│ eva                       │ VARCHAR      │ YES     │ NULL    │ NULL    │ NULL    │
│ train_number              │ VARCHAR      │ YES     │ NULL    │ NULL    │ NULL    │
│ line_number               │ VARCHAR      │ YES     │ NULL    │ NULL    │ NULL    │
│ final_destination_station │ VARCHAR      │ YES     │ NULL    │ NULL    │ NULL    │
│ delay_in_min              │ INTEGER      │ YES     │ NULL    │ NULL    │ NULL    │
│ time                      │ TIMESTAMP_NS │ YES     │ NULL    │ 

In [10]:
duckdb.sql("SELECT * FROM frankfurt LIMIT 20").df()

,station_name,xml_station_name,eva,train_number,line_number,final_destination_station,delay_in_min,time,arrival_is_canceled,departure_is_canceled,...,train_line_ride_id,train_line_station_num,arrival_planned_time,arrival_change_time,departure_planned_time,departure_change_time,id,datum,wochentag,zuggruppe
0,Frankfurt (Main) Hbf,Frankfurt Hbf (tief),08098105,35647,6,Langen(Hess),1,2025-10-10 16:03:00,False,False,...,-1728853340833921709,16,2025-10-10 16:00:00,2025-10-10 16:02:00,2025-10-10 16:02:00,2025-10-10 16:03:00,-1728853340833921709-2510101520-16,2025-10-10,Friday,S-Bahn
1,Frankfurt (Main) Hbf,Frankfurt(Main)Hbf,08000105,15650,61,Frankfurt (Main) Hbf,0,2025-10-10 16:03:00,True,False,...,8800184759870349563,12,2025-10-10 16:03:00,2025-10-10 16:03:00,NaT,NaT,8800184759870349563-2510101513-12,2025-10-10,Friday,Regionalverkehr
2,Frankfurt (Main) Hbf,Frankfurt(Main)Hbf,08000105,36852,8,Frankfurt(M) Flughafen Regionalbf,0,2025-10-10 16:04:00,False,True,...,6023931554467202128,1,NaT,NaT,2025-10-10 16:04:00,2025-10-10 16:04:00,6023931554467202128-2510101604-1,2025-10-10,Friday,S-Bahn
3,Frankfurt (Main) Hbf,Frankfurt(Main)Hbf,08000105,721,NaN,München Hbf,10,2025-10-10 16:04:00,False,False,...,-388013222747031970,6,2025-10-10 15:48:00,2025-10-10 15:58:00,2025-10-10 15:54:00,2025-10-10 16:04:00,-388013222747031970-2510101354-6,2025-10-10,Friday,Fernverkehr
4,Frankfurt (Main) Hbf,Frankfurt(Main)Hbf,08000105,25021,RB10,Frankfurt (Main) Hbf,1,2025-10-10 16:05:00,False,False,...,-3735070364612616730,26,2025-10-10 16:04:00,2025-10-10 16:05:00,NaT,NaT,-3735070364612616730-2510101328-26,2025-10-10,Friday,Regionalverkehr
5,Frankfurt (Main) Hbf,Frankfurt(Main)Hbf,08000105,35750,7,Frankfurt (Main) Hbf,1,2025-10-10 16:06:00,False,False,...,-7631934461582025976,10,2025-10-10 16:05:00,2025-10-10 16:06:00,NaT,NaT,-7631934461582025976-2510101522-10,2025-10-10,Friday,S-Bahn
6,Frankfurt (Main) Hbf,Frankfurt Hbf (tief),08098105,35548,5,Friedrichsdorf(Taunus),2,2025-10-10 16:06:00,False,False,...,-8162103845652612851,7,2025-10-10 16:03:00,2025-10-10 16:04:00,2025-10-10 16:04:00,2025-10-10 16:06:00,-8162103845652612851-2510101553-7,2025-10-10,Friday,S-Bahn
7,Frankfurt (Main) Hbf,Frankfurt(Main)Hbf,08000105,15552,51,Bad Soden-Salmünster,1,2025-10-10 16:07:00,False,False,...,-2438566548480237825,1,NaT,NaT,2025-10-10 16:06:00,2025-10-10 16:07:00,-2438566548480237825-2510101606-1,2025-10-10,Friday,Regionalverkehr
8,Frankfurt (Main) Hbf,Frankfurt(Main)Hbf,08000105,15019,RB41,Frankfurt (Main) Hbf,0,2025-10-10 16:07:00,True,False,...,-3806743340109451607,19,2025-10-10 16:07:00,2025-10-10 16:07:00,NaT,NaT,-3806743340109451607-2510101429-19,2025-10-10,Friday,Regionalverkehr
9,Frankfurt (Main) Hbf,Frankfurt(Main)Hbf,08000105,15119,RB40,Frankfurt (Main) Hbf,0,2025-10-10 16:07:00,True,False,...,-6447652391797122851,18,2025-10-10 16:07:00,2025-10-10 16:07:00,NaT,NaT,-6447652391797122851-2510101431-18,2025-10-10,Friday,Regionalverkehr


In [11]:
duckdb.sql("""
     SELECT * FROM frankfurt
     ORDER BY delay_in_min DESC
     LIMIT 10
""").df()

,station_name,xml_station_name,eva,train_number,line_number,final_destination_station,delay_in_min,time,arrival_is_canceled,departure_is_canceled,...,train_line_ride_id,train_line_station_num,arrival_planned_time,arrival_change_time,departure_planned_time,departure_change_time,id,datum,wochentag,zuggruppe
0,Frankfurt (Main) Hbf,Frankfurt(Main)Hbf,08000105,403,NaN,Zürich HB,174,2025-10-12 05:39:00,True,True,...,-5851721557238592488,8,2025-10-12 02:20:00,2025-10-12 05:22:00,2025-10-12 02:45:00,2025-10-12 05:39:00,-5851721557238592488-2510111915-8,2025-10-12,Sunday,Fernverkehr
1,Frankfurt (Main) Hbf,Frankfurt(Main)Hbf,08000105,60403,NaN,Zürich HB,172,2025-10-12 05:37:00,False,False,...,7582314568044505717,8,2025-10-12 02:20:00,2025-10-12 05:20:00,2025-10-12 02:45:00,2025-10-12 05:37:00,7582314568044505717-2510111915-8,2025-10-12,Sunday,Fernverkehr
2,Frankfurt (Main) Hbf,Frankfurt(Main)Hbf,08000105,618,NaN,Kiel Hbf,134,2025-10-10 07:00:00,False,False,...,2686007473625185344,6,2025-10-10 04:38:00,2025-10-10 06:10:00,2025-10-10 04:46:00,2025-10-10 07:00:00,2686007473625185344-2510092313-6,2025-10-10,Friday,Fernverkehr
3,Frankfurt (Main) Hbf,Frankfurt(Main)Hbf,08000105,15442,67,Frankfurt (Main) Hbf,129,2025-10-12 01:57:00,False,False,...,-4369812750746541737,15,2025-10-11 23:48:00,2025-10-12 01:57:00,NaT,NaT,-4369812750746541737-2510112246-15,2025-10-12,Sunday,Regionalverkehr
4,Frankfurt (Main) Hbf,Frankfurt(Main)Hbf,08000105,27,NaN,Wien Hbf,128,2025-10-10 14:30:00,False,False,...,-1790024725878292801,11,2025-10-10 12:12:00,2025-10-10 14:25:00,2025-10-10 12:22:00,2025-10-10 14:30:00,-1790024725878292801-2510100823-11,2025-10-10,Friday,Fernverkehr
5,Frankfurt (Main) Hbf,Frankfurt(Main)Hbf,08000105,403,NaN,Zürich HB,120,2025-10-07 04:45:00,False,False,...,-5851721557238592488,9,2025-10-07 02:20:00,2025-10-07 03:42:00,2025-10-07 02:45:00,2025-10-07 04:45:00,-5851721557238592488-2510061915-9,2025-10-07,Tuesday,Fernverkehr
6,Frankfurt (Main) Hbf,Frankfurt(Main)Hbf,08000105,112,NaN,Frankfurt (Main) Hbf,114,2025-10-07 21:34:00,False,False,...,2979813652439599907,23,2025-10-07 19:40:00,2025-10-07 21:34:00,NaT,NaT,2979813652439599907-2510071042-23,2025-10-07,Tuesday,Fernverkehr
7,Frankfurt (Main) Hbf,Frankfurt(Main)Hbf,08000105,9553,NaN,Frankfurt (Main) Hbf,108,2025-10-09 18:47:00,False,False,...,2182708515882328076,5,2025-10-09 16:59:00,2025-10-09 18:47:00,NaT,NaT,2182708515882328076-2510091310-5,2025-10-09,Thursday,Fernverkehr
8,Frankfurt (Main) Hbf,Frankfurt(Main)Hbf,08000105,60403,NaN,Zürich HB,106,2025-10-07 04:31:00,False,False,...,7582314568044505717,9,2025-10-07 02:20:00,2025-10-07 03:42:00,2025-10-07 02:45:00,2025-10-07 04:31:00,7582314568044505717-2510061915-9,2025-10-07,Tuesday,Fernverkehr
9,Frankfurt (Main) Hbf,Frankfurt(Main)Hbf,08000105,4634,54,Frankfurt (Main) Hbf,104,2025-10-09 01:28:00,False,False,...,1556741539828157941,32,2025-10-08 23:44:00,2025-10-09 01:28:00,NaT,NaT,1556741539828157941-2510082135-32,2025-10-09,Thursday,Regionalverkehr


In [12]:
duckdb.sql("""
     SELECT delay_in_min, COUNT(*)AS anzahl
     FROM frankfurt
     WHERE delay_in_min < 0
     GROUP BY delay_in_min
     ORDER BY delay_in_min
""")

┌──────────────┬────────┐
│ delay_in_min │ anzahl │
│    int32     │ int64  │
├──────────────┼────────┤
│           -8 │      1 │
│           -7 │      3 │
│           -6 │      2 │
│           -5 │     13 │
│           -4 │     15 │
│           -3 │     32 │
│           -2 │     69 │
│           -1 │    245 │
└──────────────┴────────┘

In [13]:
duckdb.sql("""
     PIVOT (
         SELECT train_type, (-delay_in_min) || 'Min. zu fruh' AS minuten 
         FROM frankfurt 
         WHERE delay_in_min < 0
     )

     ON minuten
     USING COUNT(*)
     GROUP BY train_type
     ORDER BY train_type
""").df()

,train_type,1Min. zu fruh,2Min. zu fruh,3Min. zu fruh,4Min. zu fruh,5Min. zu fruh,6Min. zu fruh,7Min. zu fruh,8Min. zu fruh
0,ECE,3,0,0,0,0,0,0,0
1,HLB,7,3,5,6,3,0,1,1
2,IC,1,0,0,1,2,0,0,0
3,ICE,39,18,5,1,3,1,1,0
4,RB,61,16,6,1,0,0,0,0
5,RE,63,20,9,3,2,0,0,0
6,S,43,1,0,0,0,0,0,0
7,STN,2,2,0,0,0,0,0,0
8,VIA,26,9,7,3,3,1,1,0


**Pivot-Tabelle: zu frühe Halte nach Zugtyp**

So liest man die Tabelle:
- **Zeilen:** Zugtyp
- **Spalten:** wie viele Minuten der Zug zu früh war
- **Werte:** Anzahl der Halte

Beispiel: In der Zeile `ICE` und der Spalte für 2 Minuten steht 18. Das heißt: 18 ICE-Halte waren 2 Minuten zu früh.

**Was die Tabelle zeigt**
- Die meisten Züge waren nur **1 Minute** zu früh.
- Die S-Bahn war fast nie mehr als 1 Minute zu früh (43 von 44 Halten).
- Der größte Wert ist **8 Minuten** (1 Halt, HLB).

In [14]:
duckdb.sql("""
       SELECT datum, wochentag, COUNT(*) AS anzahl
       FROM frankfurt
       GROUP BY datum, wochentag
       ORDER BY datum
""")

┌────────────┬───────────┬────────┐
│   datum    │ wochentag │ anzahl │
│    date    │  varchar  │ int64  │
├────────────┼───────────┼────────┤
│ 2025-10-06 │ Monday    │   1927 │
│ 2025-10-07 │ Tuesday   │   1942 │
│ 2025-10-08 │ Wednesday │   1951 │
│ 2025-10-09 │ Thursday  │   1955 │
│ 2025-10-10 │ Friday    │   1935 │
│ 2025-10-11 │ Saturday  │   1717 │
│ 2025-10-12 │ Sunday    │   1628 │
└────────────┴───────────┴────────┘

## 4. Frage 1: Wie viele Züge sind pünktlich?

Definition der Deutschen Bahn: Ein Zug ist pünktlich, wenn er weniger als 6 Minuten Verspätung hat (`delay_in_min <= 5`). Ausgefallene Halte werden nicht mitgezählt.


In [15]:
duckdb.sql("""
    SELECT 
      COUNT(*) AS gesamt,
      SUM(CASE WHEN delay_in_min <= 5 THEN 1 ELSE 0 END) AS puenktlich,
      ROUND(100.0 * SUM(CASE WHEN delay_in_min <= 5 THEN 1 ELSE 0 END) / COUNT(*) , 1) AS prozent
    FROM frankfurt
    WHERE NOT arrival_is_canceled
      AND NOT departure_is_canceled
        
""")

┌────────┬────────────┬─────────┐
│ gesamt │ puenktlich │ prozent │
│ int64  │   int128   │ double  │
├────────┼────────────┼─────────┤
│  12503 │       9494 │    75.9 │
└────────┴────────────┴─────────┘

In [16]:
duckdb.sql("""
     SELECT 
       datum,
       wochentag,
       COUNT(*) AS gesamt,
       COUNT_IF(delay_in_min <= 5 ) AS puenktlich,
       COUNT_IF(delay_in_min >= 6 ) AS verspeatet,
       ROUND(100.0 * SUM(CASE WHEN delay_in_min <= 5 THEN 1 ELSE 0 END) / COUNT(*) ,1) AS prozent_pkt
     FROM frankfurt
     WHERE NOT arrival_is_canceled
       AND NOT departure_is_canceled
     GROUP BY datum, wochentag
     ORDER BY datum    
""").df()

,datum,wochentag,gesamt,puenktlich,verspeatet,prozent_pkt
0,2025-10-06,Monday,1828,1284.0,544.0,70.2
1,2025-10-07,Tuesday,1868,1395.0,473.0,74.7
2,2025-10-08,Wednesday,1830,1307.0,523.0,71.4
3,2025-10-09,Thursday,1880,1389.0,491.0,73.9
4,2025-10-10,Friday,1823,1351.0,472.0,74.1
5,2025-10-11,Saturday,1697,1382.0,315.0,81.4
6,2025-10-12,Sunday,1577,1386.0,191.0,87.9


**Ergebnis Frage 1**
- In der Woche waren **75,9 %** der Halte pünktlich. Etwa jeder vierte Zug hatte 6 Minuten oder mehr Verspätung.
- Zusätzlich sind **4,2 %** der Halte ausgefallen.
- Am schlechtesten war **Montag** (70,2 %), am besten **Sonntag** (87,9 %).
- Unter der Woche liegt die Pünktlichkeit bei 70–75 %, am Wochenende bei 81–88 %.
- Mögliche Erklärung (Hypothese): Unter der Woche fahren mehr Züge (ca. 1.930–1.955 pro Tag, am Wochenende 1.628–1.717). Verspätungen übertragen sich dann leichter. Die Daten zeigen den Grund aber nicht.
